In [ ]:

import pandas as pd
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
# Задание 1
airports = pd.read_csv(
    'https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat',
    header=None,
    names=["Airport ID", "Name", "City", "Country", "IATA", "ICAO", "Latitude", "Longitude", "Altitude", "Timezone", "DST", "Tz database time zone", "Type", "Source"]
)
routes = pd.read_csv(
    'https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat',
    header=None,
    names=["Airline", "Airline ID", "Source airport", "Source airport ID", "Destination airport", "Destination airport ID", "Codeshare", "Stops", "Equipment"]
)

# Фильтруем маршруты
routes = routes[(routes["Source airport ID"] != "\\N") & (routes["Destination airport ID"] != "\\N")]
routes["Source airport ID"] = routes["Source airport ID"].astype(int)
routes["Destination airport ID"] = routes["Destination airport ID"].astype(int)

# Считаем взлеты и посадки
flights = pd.DataFrame({
    "takeoffs": routes["Source airport ID"].value_counts(),
    "landings": routes["Destination airport ID"].value_counts()
}).fillna(0)
flights["sum"] = flights["takeoffs"] + flights["landings"]

# Объединяем с данными аэропортов
top10 = flights.merge(
    airports[["Airport ID", "Name", "Country"]],
    left_index=True,
    right_on="Airport ID"
).sort_values(by="sum", ascending=False).head(10)

# Формируем мультииндекс
top10_final = top10.set_index(["Country", "Name"])[["takeoffs", "landings", "sum"]]

# Выводим результат
print(top10_final)

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>

In [ ]:
# Задание 2
# Проблемы с пропущенными значениями
missing_airports = airports[airports.isnull().any(axis = 1)]
missing_routes = routes[routes.isnull().any(axis = 1)]

# Проблемы с типами данных
invalid_routes = routes[routes["Source airport ID"].isnull() | routes["Destination airport ID"].isnull()]

# Дубликаты
duplicate_airports = airports[airports.duplicated()]
duplicate_routes = routes[routes.duplicated()]

# Несоответствия данных 
mismatched_airports = routes[~routes["Source airport ID"].isin(airports["Airport ID"]) | ~routes["Destination airport ID"].isin(airports["Airport ID"])]

# Логические ошибки 
flights = pd.DataFrame({
    "takeoffs": routes["Source airport ID"].value_counts(),
    "landings": routes["Destination airport ID"].value_counts()
}).fillna(0)
flights["sum"] = flights["takeoffs"] + flights["landings"]
invalid_flights = flights[(flights["takeoffs"] == 0) | (flights["landings"] == 0)]

# Выводим результаты
print(f"Пропущенные значения в аэропортах: {missing_airports.shape[0]} строк")
print(f"Пропущенные значения в маршрутах: {missing_routes.shape[0]} строк")

print(f"Некорректные маршруты с ошибками в ID: {invalid_routes.shape[0]} строк")

print(f"Дубликаты в данных о аэропортах: {duplicate_airports.shape[0]}")
print(f"Дубликаты в данных о маршрутах: {duplicate_routes.shape[0]}")

print(f"Несоответствующие аэропорты: {mismatched_airports.shape[0]} строк")

print(f"Логические ошибки в данных о взлетах и посадках: {invalid_flights.shape[0]} строк")

NameError: name 'airports' is not defined

Задание 2
в этой части мы ищем и выводим все ошибки, такие как пропущенные значения - сначал мы пллучаем датафейм со значениями true для пропущенных и falce для непустых, после any(axis = 1) проверяет по строкам нет ли NaN,
неправильные типы данных - также проверяем через isnull() есть ли пропущенные значения в итоге получим все строки где есть пропущенные значения,
дубликаты - через duplicated() проверяем есть ли в датафрейме строки дубликаты и мы получим true если они есть,
несоответсвия - используя insin() проверяем есть ли элемент из одного столбца в другом, так мы можем найти маршруты с аэропортами котрых нет в базе данных аэропортов,
логические ошибки - используем value.counts() который подсчитыввает уникальные значения в столбце, но в нашем случае подсчитывает сколько каждый аэропорт появляется в качестве источника маршрута и когда аэропорт появляется в качестве пункта назначения,  заменяем все NaN на 0, так как value.counts() может не посчитать аэропорты если они не встречаются в столбце
и в конце выводим результаты и добавляем shape[0] чтобы получитьь количеств строк в фрейме и возвращает кортеж строки стобцы и мы берем тольок строки

In [ ]:
# Задание 3
# Создание словаря континентов для стран
continent_dict = {
    "Europe": ["Germany", "France", "UK", "Italy", "Spain", "Russia", "Ukraine"],
    "Asia": ["China", "India", "Japan", "South Korea", "Turkey", "Iran", "Thailand"],
    "Africa": ["Egypt", "South Africa", "Nigeria", "Kenya", "Ethiopia"],
    "North America": ["United States", "Canada", "Mexico"],
    "South America": ["Brazil", "Argentina", "Chile", "Colombia", "Peru"],
    "Australia": ["Australia"],
    "Antarctica": ["Antarctica"]
}

def get_continent(country):
    for continent, countries in continent_dict.items():
        if country in countries:
            return continent
    return None  # Если страна не найдена, везвращает None

# Добавление континента в таблицу аэропортов
airports['Continent'] = airports['Country'].apply(get_continent)

# Обработка маршрутов и очистка данных
routes["Source airport ID"] = pd.to_numeric(routes["Source airport ID"], errors='coerce')
routes["Destination airport ID"] = pd.to_numeric(routes["Destination airport ID"], errors='coerce')
routes = routes.dropna(subset=["Source airport ID", "Destination airport ID"])

# Считаем взлеты и посадки
flights = pd.DataFrame({
    "takeoffs": routes["Source airport ID"].value_counts(),
    "landings": routes["Destination airport ID"].value_counts()
}).fillna(0)
flights["sum"] = flights["takeoffs"] + flights["landings"]

# Добавляем информацию о континенте в данные о аэропортах
top10 = flights.merge(
    airports[["Airport ID", "Name", "Country", "Continent"]],
    left_index=True,
    right_on="Airport ID"
).sort_values(by="sum", ascending=False).head(10)


top10_final = top10.set_index(["Continent", "Country", "Name"])[["takeoffs", "landings", "sum"]]

# Задание 3.1: Визуализация всех аэропортов на континентах
def plot_airports_by_continent(continent):
    continent_airports = airports[airports['Continent'] == continent]

    fig = px.scatter_geo(
        continent_airports,
        lat='Latitude',
        lon='Longitude',
        hover_name='Name',
        color='Continent',
        title=f"AIRPORTS IN {continent.upper()}",
        projection="natural earth"
    )

    fig.show()

# Задание 3.2: Топ-5 крупнейших аэропортов по сумме взлетов/посадок для каждого континента и их взаимосвязи
def plot_top5_airports_with_connections(continent):
    # Выбираем топ-5 крупнейших аэропортов по сумме взлетов и посадок
    continent_airports = top10[top10['Continent'] == continent]
    top5_airports = continent_airports.nlargest(5, 'sum')

    # Строим карту для этих аэропортов
    fig = go.Figure(go.Scattergeo(
        locationmode='ISO-3',
        lat=top5_airports['Latitude'],
        lon=top5_airports['Longitude'],
        text=top5_airports['Name'],
        marker=dict(size=10, color='red')
    ))

    # Добавляем линии для маршрутов между аэропортами
    for _, route in routes.iterrows():
        source = route['Source airport ID']
        destination = route['Destination airport ID']
        if source in top5_airports.index and destination in top5_airports.index:
            # Проверка односторонней или двусторонней связи
            is_bidirectional = source in routes[routes['Destination airport ID'] == destination].index
            color = 'red' if is_bidirectional else 'green'

            fig.add_trace(go.Scattergeo(
                locationmode='ISO-3',
                lon=[top5_airports.loc[source, 'Longitude'], top5_airports.loc[destination, 'Longitude']],
                lat=[top5_airports.loc[source, 'Latitude'], top5_airports.loc[destination, 'Latitude']],
                mode='lines',
                line=dict(width=2, color=color),
                opacity=0.5
            ))

    # Добавляем описание и отображаем карту
    fig.update_layout(
        geo=dict(projection_type="natural earth"),
        title=f"Top 5 Airports in {continent} with Connections"
    )

    fig.show()

# Пример использования для континента - можем в кавычки вставить любой из континентов
continent = ''
plot_airports_by_continent(continent)
plot_top5_airports_with_connections(continent)

Задание 3
создаем словарь, где ключи это континенты, а занчения это список стран принадлежащих этим континентам. get_continent(country) - принимает название страны и возвращает соответсвующий континент.
дальше создаем новый столбец в который хаписываем континент, еоторый возвращает функция, и apply() нуужен чтобы применить это для каждого элемента этого столбца.
дальше преобразуем Source airport ID и Destination airport ID в числовой формат, чтобы некоректные значения превратились в NaN, удалем строки спропущенными значениями для корректного расчета и затем создаем таблицу в которой считаем количество взлетов и посадок
дальше соединяем  данные о взлетах и посадках с помощью merge(), потом сортируем в порядке убывания и берем 10 аэропортов
дальше в датафрейме оставляем только нужные столбцы 

Задание 3.1
с помощью функции создаем карту, отображаем аэропорты и добавляем их название
Задание 3.2
отображаем 5 крупнейших аэропортов для заданного континента, можно поставить люой континент и эти две функции покажут в первом случае все аэропорты по этому континенты, во втором 5 крупнейших и их взаимосвязи